# 90—Publish snapshot to Cloud Storage

### ⚠️ ROI maintainers only. This is not a student notebook.

It writes to an ROI-owned bucket and will throw a permissions error for anyone else. It is in
the repo because it belongs next to the thing it mirrors, not because students should run it.

## The important design decision

**This notebook does not reimplement the transformations.** It fetches
`01_load_explore.ipynb` by raw URL, strips the cells that write to BigQuery, executes the rest
once per metro with `METRO` overridden, and exports the resulting dataframes to Parquet.

That guarantees the snapshot cannot drift from what students actually get. The moment someone
edits the student notebook, this picks the change up on the next run. A hand-maintained copy
of the same logic would be wrong within a week.

## Why the snapshot exists at all

`01_load_explore.ipynb` pulls live from irs.gov, foodsafety.gov, and Seattle's Socrata API.
Three external dependencies, on a day when 150 people hit them inside the same ten minutes.
If any one is down or rate-limiting, the notebook fails for the entire room at once.

`scripts/load.sh` rebuilds every table from this snapshot instead, so a single upstream outage
costs a team five minutes rather than their afternoon.

## Bucket layout

```
gs://class-demo/a4i-2026/challenge-2-food-equity/<metro>/<table>/*.parquet
```

`a4i-2026` is the generic top level every challenge shares.
The bucket needs `allUsers:objectViewer` so students can read anonymously from their own
projects—`load.sh` does not authenticate against it.

In [ ]:
# --- Configuration ---------------------------------------------------------
BUCKET  = "class-demo"
PREFIX  = "a4i-2026/challenge-2-food-equity"

NOTEBOOK_URL = ("https://raw.githubusercontent.com/haggman/"
                "A4I2026-challenge-2-food-equity/main/notebooks/01_load_explore.ipynb")

# Every metro we publish. Adding one here is the only change needed - load.sh
# discovers what exists by listing the bucket.
METROS = ["Seattle", "Philadelphia", "Atlanta", "Chicago", "Houston", "Denver", "New York"]

TABLES = {
    "recipients":         "out",              # variable name in the student notebook
    "surplus_postings":   "surplus",
    "shelf_life":         "shelf_life",
    "tract_demographics": "tracts",
}

import google.auth
credentials, PROJECT_ID = google.auth.default()
print(f"Publishing from project: {PROJECT_ID}")
print(f"Target: gs://{BUCKET}/{PREFIX}/<metro>/<table>/")

## Fetch the student notebook and extract its code

We take the code cells, drop the `%%bigquery` magics (they need a live dataset and we are not
loading anything here), and drop the cell that calls `load_table`—the whole point is to export
dataframes rather than write BigQuery tables.

In [ ]:
import re
import requests
import nbformat

nb = nbformat.reads(requests.get(NOTEBOOK_URL, timeout=60).text, as_version=4)

sources = []
for cell in nb.cells:
    if cell.cell_type != "code":
        continue
    src = cell.source
    if src.lstrip().startswith("%%"):          # %%bigquery cells need loaded tables
        continue
    if "load_table(" in src:                   # skip the BigQuery write section
        src = re.sub(r"^\s*load_table\(.*?\)\s*$", "", src, flags=re.M | re.S)
    sources.append(src)

print(f"Code cells in the student notebook : {sum(1 for c in nb.cells if c.cell_type == 'code')}")
print(f"Cells we will execute              : {len(sources)}")

## Build and export, one metro at a time

Each metro runs in its own namespace so a failure in one cannot contaminate the next. We keep
going on failure and report at the end—one bad metro should not cost you the other six.

In [ ]:
import io
import time
import traceback
import pandas as pd
from google.cloud import storage

gcs = storage.Client(project=PROJECT_ID)
bucket = gcs.bucket(BUCKET)

results = {}

for metro in METROS:
    print(f"\n{'=' * 64}\n{metro}\n{'=' * 64}")
    t0 = time.time()
    ns = {"__name__": "__main__"}
    try:
        for i, src in enumerate(sources):
            # Override the metro on the config cell only.
            if "METRO = " in src and "METROS = " in src:
                src = re.sub(r'^METRO\s*=\s*".*?"', f'METRO = "{metro}"', src, count=1, flags=re.M)
            exec(compile(src, f"<cell {i}>", "exec"), ns)

        slug = metro.lower().replace(" ", "-")
        for table, var in TABLES.items():
            df = ns.get(var)
            if df is None or not isinstance(df, pd.DataFrame):
                raise RuntimeError(f"expected dataframe {var!r} for table {table!r}")
            df = df.drop(columns=[c for c in ("tract_geom",) if c in df.columns])
            buf = io.BytesIO()
            df.to_parquet(buf, index=False)
            buf.seek(0)
            blob = bucket.blob(f"{PREFIX}/{slug}/{table}/data.parquet")
            blob.upload_from_file(buf, content_type="application/octet-stream")
            print(f"  {table:<22} {len(df):>7,} rows -> gs://{BUCKET}/{blob.name}")

        results[metro] = ("OK", time.time() - t0)
    except Exception as exc:                              # noqa: BLE001
        print(f"  FAILED: {exc}")
        traceback.print_exc()
        results[metro] = (f"FAILED: {exc}", time.time() - t0)

print(f"\n{'=' * 64}\nSUMMARY\n{'=' * 64}")
for metro, (status, secs) in results.items():
    print(f"  {metro:<16} {secs:>6.1f}s  {status}")

## Verify the snapshot is loadable

Publishing is not the same as publishing something that works. This reads every file back the
way `load.sh` will, and checks the one number that decides whether vector search can work at
all—profile variety. A snapshot where every profile is identical would load cleanly and be
useless.

In [ ]:
ok = True
for metro in METROS:
    slug = metro.lower().replace(" ", "-")
    for table in TABLES:
        path = f"{PREFIX}/{slug}/{table}/data.parquet"
        blob = bucket.blob(path)
        if not blob.exists():
            print(f"  MISSING  gs://{BUCKET}/{path}")
            ok = False
            continue
        df = pd.read_parquet(io.BytesIO(blob.download_as_bytes()))
        note = ""
        if table == "recipients":
            variety = df["profile_text"].nunique() / max(len(df), 1)
            note = f"  profile variety {variety:.1%}"
            if variety < 0.9:
                note += "   <-- TOO HOMOGENEOUS, REGENERATE"
                ok = False
        print(f"  {metro:<14} {table:<20} {len(df):>7,} rows{note}")

print("\nAll good." if ok else "\nProblems above. Do not announce this snapshot.")